# 01 — Exploratory Data Analysis (MovieLens 1M)

This notebook explores the MovieLens 1M dataset:
- Dataset loading and schema inspection
- Rating distributions
- User and movie activity
- Sparsity analysis
- Genre breakdown
- Temporal patterns

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR, RATINGS_COLS, MOVIES_COLS, USERS_COLS
from src.logging_utils import data_logger

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
data_logger.start_phase('eda', 'EDA notebook started')

## 1. Load Data

In [ ]:
ratings = pd.read_parquet(PROCESSED_DIR / 'ratings.parquet')
movies  = pd.read_parquet(PROCESSED_DIR / 'movies.parquet')
users   = pd.read_parquet(PROCESSED_DIR / 'users.parquet')

print(f'Ratings: {len(ratings):,} rows')
print(f'Movies:  {len(movies):,} rows')
print(f'Users:   {len(users):,} rows')
ratings.head()

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of rating values
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Rating Value Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings per user
user_counts = ratings.groupby('user_id').size()
user_counts.hist(bins=50, ax=axes[1])
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('User Count')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f'Mean rating:       {ratings["rating"].mean():.3f}')
print(f'Std dev:           {ratings["rating"].std():.3f}')
print(f'Mean ratings/user: {user_counts.mean():.1f}')
print(f'Median ratings/user: {user_counts.median():.1f}')

## 3. Sparsity

In [ ]:
n_users  = ratings['user_id'].nunique()
n_movies = ratings['movie_id'].nunique()
n_ratings = len(ratings)
sparsity = 1.0 - n_ratings / (n_users * n_movies)

print(f'Unique users:  {n_users:,}')
print(f'Unique movies: {n_movies:,}')
print(f'Total ratings: {n_ratings:,}')
print(f'Matrix size:   {n_users * n_movies:,}')
print(f'Sparsity:      {sparsity:.4%}')

## 4. Movie Popularity (Long-tail Distribution)

In [ ]:
movie_counts = ratings.groupby('movie_id').size().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
plt.plot(range(len(movie_counts)), movie_counts.values)
plt.title('Movie Popularity (Long-tail)')
plt.xlabel('Movie rank')
plt.ylabel('Number of ratings')
plt.yscale('log')
plt.tight_layout()
plt.show()

top10 = movie_counts.head(10)
top10_movies = movies.set_index('movie_id').loc[top10.index, 'title']
print('Top 10 most-rated movies:')
for mid, title in top10_movies.items():
    print(f'  {title}: {top10[mid]:,} ratings')

## 5. Genre Distribution

In [ ]:
genre_cols = [c for c in movies.columns if c.startswith('genre_')]
genre_counts = movies[genre_cols].sum().sort_values(ascending=False)
genre_counts.index = [c.replace('genre_', '') for c in genre_counts.index]

plt.figure(figsize=(12, 5))
genre_counts.plot(kind='bar')
plt.title('Movies per Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Temporal Analysis

In [ ]:
if 'datetime' not in ratings.columns:
    ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

monthly = ratings.set_index('datetime').resample('ME').size()

plt.figure(figsize=(14, 4))
monthly.plot()
plt.title('Ratings per Month')
plt.xlabel('Date')
plt.ylabel('Number of Ratings')
plt.tight_layout()
plt.show()

data_logger.end_phase('eda', 'EDA notebook complete')